# 000 · Programming fundamentals — procedural layer

**Domain mnemonic — 0 is the empty origin:** null, zero, the blank slate every program starts from.

Codes covered: **008** implicit type coercion · **018** shadowing · **023** short-circuit evaluation · **037** nesting depth · **046** off-by-one error · **057** text vs bytes · **063** the 0.1 + 0.2 problem · **079** re-raise & exception chaining · **083** with statement (context manager) · **099** aliasing.

**Method:** read the cell → **predict the output** → run it → compare. The matrix holds the imagery; this notebook engraves the net effect. Cells run top to bottom in one kernel. Three cells raise on purpose — their first line says so, and the traceback itself is the output to study.

## 008 · Implicit type coercion

Coercion is the language auto-converting a value's type in mixed expressions: `1 + 2.0 -> 3.0`; JS `"1" + 1 -> "11"`.

*A pushy customs officer at the border between number-land and string-land restamps your passport to the other nationality without asking - you only notice when "11" comes out.*

Watch: the result type flips with no call from you — and Python refuses the string+int crossing that JS silently allows.

In [1]:
# Mixed expressions: who restamped the passport?
a = 1 + 2.0
print("1 + 2.0      ->", repr(a), "| type:", type(a).__name__)
b = True + 1
print("True + 1     ->", repr(b), "| type:", type(b).__name__, "(bool coerced to int)")
try:
    "1" + 1
except TypeError as e:
    print('"1" + 1      -> TypeError:', e)
print('str(1) + "1" ->', repr(str(1) + "1"), "| explicit conversion: you called str(), nothing implicit")

1 + 2.0      -> 3.0 | type: float
True + 1     -> 2 | type: int (bool coerced to int)
"1" + 1      -> TypeError: can only concatenate str (not "int") to str
str(1) + "1" -> '11' | explicit conversion: you called str(), nothing implicit


## 018 · Shadowing

Shadowing is an inner-scope variable reusing an outer name, hiding the outer binding until the inner scope ends.

*An eclipse: the small inner moon named x slides in front of the huge outer sun named x. The sun still burns behind it, untouched and invisible, until the moon moves on.*

Watch: inside the function you see the moon; after it returns, the outer sun is still exactly what it was.

In [2]:
x = "outer sun"

def eclipse():
    x = "inner moon"  # new binding, not a reassignment of the outer x
    print("inside eclipse(): x =", repr(x))

eclipse()
print("after eclipse() : x =", repr(x), "(outer binding untouched)")

inside eclipse(): x = 'inner moon'
after eclipse() : x = 'outer sun' (outer binding untouched)


## 023 · Short-circuit evaluation

`and`/`or` stop evaluating once the outcome is known: `False and f()` never calls `f` - enabling guards like `x != 0 and 1/x`.

*A circuit breaker that trips at the first False: current never reaches the second bulb, so it stays dark and its side effects never fire at all.*

Watch: which bulbs print "lit" — after a tripped `and`, the right-hand side never runs, so its side effect never happens.

In [3]:
def bulb(label):
    print("  side effect: bulb", label, "lit")
    return True

print("True and bulb('A') :")
result = True and bulb("A")
print("  ->", result)
print("False and bulb('B'):")
result = False and bulb("B")
print("  ->", result, "(breaker tripped: bulb B never evaluated)")

x = 0
safe = x != 0 and 10 / x > 1
print("x = 0; x != 0 and 10/x > 1 ->", safe, "(division skipped, no ZeroDivisionError)")

True and bulb('A') :
  side effect: bulb A lit
  -> True
False and bulb('B'):
  -> False (breaker tripped: bulb B never evaluated)
x = 0; x != 0 and 10/x > 1 -> False (division skipped, no ZeroDivisionError)


## 037 · Nesting depth

Deeply nested conditionals (the arrow anti-pattern) hide logic; flatten with guard clauses, early returns, or extracted functions.

*Code shaped like an arrowhead of Russian dolls: to reach the else at the tip you pry open five dolls, and by the third you have forgotten what the first one held.*

Watch: both functions print identical results on all 8 inputs — the guard-clause version does it at one indentation level instead of four.

In [4]:
def nested(a, b, c):          # the arrowhead
    if a:
        if b:
            if c:
                return "do()"
            else:
                return "stopped: no c"
        else:
            return "stopped: no b"
    else:
        return "stopped: no a"

def flat(a, b, c):            # guard-clause inversion
    if not a:
        return "stopped: no a"
    if not b:
        return "stopped: no b"
    if not c:
        return "stopped: no c"
    return "do()"

cases = [(a, b, c) for a in (False, True) for b in (False, True) for c in (False, True)]
for case in cases:
    n, f = nested(*case), flat(*case)
    assert n == f
    print(case, "-> nested:", n, "| flat:", f)
print("identical on all", len(cases), "inputs; deepest indent: 4 levels -> 1 level")

(False, False, False) -> nested: stopped: no a | flat: stopped: no a
(False, False, True) -> nested: stopped: no a | flat: stopped: no a
(False, True, False) -> nested: stopped: no a | flat: stopped: no a
(False, True, True) -> nested: stopped: no a | flat: stopped: no a
(True, False, False) -> nested: stopped: no b | flat: stopped: no b
(True, False, True) -> nested: stopped: no b | flat: stopped: no b
(True, True, False) -> nested: stopped: no c | flat: stopped: no c
(True, True, True) -> nested: do() | flat: do()
identical on all 8 inputs; deepest indent: 4 levels -> 1 level


## 046 · Off-by-one error

Fencepost errors run a loop once too many or too few by confusing counts with boundaries: 10 posts hold only 9 rails.

*A builder orders 10 rails for his 10 fence posts and stands baffled at the end, one rail dangling from his hand - posts are boundaries; the gaps are the count.*

Watch: the loop survives indexes 0..2, then dies asking for index 3 of a 3-element list — one pass too many.

In [5]:
# INTENDED ERROR — read the traceback
rails = ["rail-0", "rail-1", "rail-2"]
print("len(rails) =", len(rails), "| valid indexes: 0, 1, 2")
for i in range(len(rails) + 1):   # + 1 is the fencepost
    print("index", i, "->", rails[i])

len(rails) = 3 | valid indexes: 0, 1, 2
index 0 -> rail-0
index 1 -> rail-1
index 2 -> rail-2


IndexError: list index out of range

In [6]:
# FIX: range(len(rails)) — the half-open [0, 3): length is a count, not a last index
rails = ["rail-0", "rail-1", "rail-2"]
for i in range(len(rails)):
    print("index", i, "->", rails[i])
print("loop ran exactly", len(rails), "times — count matches length")

index 0 -> rail-0
index 1 -> rail-1
index 2 -> rail-2
loop ran exactly 3 times — count matches length


## 057 · Text vs bytes

`str` is decoded text (code points); `bytes` are raw octets; convert only at the edges with `.encode()`/`.decode()` - never mix the two.

*A border checkpoint between meaning-land and wire-land: prose must pass the encode booth to become numbered freight, and freight must pass decode to become prose. No cargo walks through untranslated.*

Watch: 'é' becomes two bytes on the wire, and the ascii booth rejects the first of them (0xc3).

In [7]:
# INTENDED ERROR — read the traceback
freight = "café".encode("utf-8")
print("text 'café' encoded ->", freight, "| type:", type(freight).__name__, "| length:", len(freight), "bytes for 4 characters")
freight.decode("ascii")   # wrong booth: ascii cannot read byte 0xc3

text 'café' encoded -> b'caf\xc3\xa9' | type: bytes | length: 5 bytes for 4 characters


UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 3: ordinal not in range(128)

In [8]:
# FIX: decode with the codec that produced the bytes
freight = "café".encode("utf-8")
prose = freight.decode("utf-8")
print("bytes", freight, "-> .decode('utf-8') ->", repr(prose), "| type:", type(prose).__name__)
print("round trip intact:", prose == "café")

bytes b'caf\xc3\xa9' -> .decode('utf-8') -> 'café' | type: str
round trip intact: True


## 063 · 0.1 + 0.2 problem

`0.1 + 0.2 == 0.30000000000000004` because 0.1 and 0.2 have no finite binary expansion - like 1/3 in decimal, rounded at parse time.

*Two measuring cups labeled 0.1 and 0.2, each secretly overfilled by a molecule at the factory (binary rounding): pour them together and the beaker needle trembles at 0.30000000000000004.*

Watch: the repr exposes the trembling needle; `==` fails, while `math.isclose` and `Decimal` behave — and the inputs, not the addition, are to blame.

In [9]:
import math
from decimal import Decimal

s = 0.1 + 0.2
print("0.1 + 0.2                     ->", repr(s))
print("0.1 + 0.2 == 0.3              ->", s == 0.3)
print("math.isclose(0.1 + 0.2, 0.3)  ->", math.isclose(s, 0.3))
print("Decimal('0.1')+Decimal('0.2') ->", Decimal("0.1") + Decimal("0.2"))
print("blame the inputs: 0.1 is really %.20f" % 0.1)

0.1 + 0.2                     -> 0.30000000000000004
0.1 + 0.2 == 0.3              -> False
math.isclose(0.1 + 0.2, 0.3)  -> True
Decimal('0.1')+Decimal('0.2') -> 0.3
blame the inputs: 0.1 is really 0.10000000000000000555


## 079 · Re-raise & exception chaining

Bare `raise` rethrows the current exception intact; `raise New(...) from e` wraps it while preserving the original cause and traceback.

*A relay runner catches a flaming baton, tapes a note to it - 'happened while loading config' - and hurls it onward; the tape ('from e') keeps the original scorch marks readable underneath.*

Watch: the traceback shows the original KeyError first, then "the direct cause of", then the new ValueError — two layers, one story.

In [10]:
# INTENDED ERROR — read the traceback
config = {"host": "localhost"}
print("config =", config, "— now asking for the missing 'port'...")
try:
    port = config["port"]
except KeyError as e:
    raise ValueError("bad config: missing 'port'") from e

config = {'host': 'localhost'} — now asking for the missing 'port'...


ValueError: bad config: missing 'port'

In [11]:
# FIX: a caller can catch the wrapper and still read the original scorch marks via __cause__
config = {"host": "localhost"}
try:
    try:
        port = config["port"]
    except KeyError as e:
        raise ValueError("bad config: missing 'port'") from e
except ValueError as err:
    print("caught    :", repr(err))
    print("__cause__ :", repr(err.__cause__), "(original preserved by 'from e')")

caught    : ValueError("bad config: missing 'port'")
__cause__ : KeyError('port') (original preserved by 'from e')


## 083 · with statement (context manager)

`with open(...) as f` guarantees `close()` on block exit - success or exception - via `__enter__`/`__exit__`; RAII's Python cousin.

*A submarine airlock: step in and the machinery arms itself; however you leave - walking out calmly or blown out by an explosion - the hatch slams and seals itself behind you.*

Watch: `f.closed` flips from False inside the block to True after it — and no one ever called `close()`.

In [12]:
import tempfile, os

fd, path = tempfile.mkstemp(suffix=".txt")
os.close(fd)

with open(path, "w") as f:
    f.write("sealed cargo")
    print("inside the airlock : f.closed =", f.closed)
print("after the block    : f.closed =", f.closed, "(hatch sealed itself — no close() call)")

with open(path) as f:
    print("read back          :", repr(f.read()))
os.remove(path)

inside the airlock : f.closed = False
after the block    : f.closed = True (hatch sealed itself — no close() call)
read back          : 'sealed cargo'


## 099 · Aliasing

Aliasing is two names bound to one object, so mutation through either shows through both; `b = a` copies nothing - use `copy()` to fork.

*One actor working two shows under two stage names: shave 'Bob' bald for the matinee and 'Robert' walks on stage bald that evening - only hiring a body double (copy) gets you two heads of hair.*

Watch: the append made through `b` shows up in `a`; only `list(a)` creates a second object that mutates independently.

In [13]:
a = [1]
b = a                # no copy: second stage name for the same actor
b.append(2)
print("after b.append(2): a =", a, "| b =", b, "| a is b:", a is b)

c = list(a)          # the body double: a real second object
c.append(99)
print("after c = list(a); c.append(99):")
print("  a =", a, "(unchanged) | c =", c, "| a is c:", a is c)

after b.append(2): a = [1, 2] | b = [1, 2] | a is b: True
after c = list(a); c.append(99):
  a = [1, 2] (unchanged) | c = [1, 2, 99] | a is c: False
